# Diagnostiquer ses données GPS

Ce notebook aide à distinguer un problème de données d'un problème de code. Il vérifie les tables attendues, les colonnes obligatoires, les utilisateurs couverts par chaque table et lance une transformation minimale sur un petit échantillon.

Principe de travail : ne pas masquer une incohérence. Si une colonne ou une phase manque, on rend le problème visible et on corrige la configuration ou les données.

## 1. Préparer l'environnement

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)

# En développement local, on importe le code source du dépôt.
# Après publication, `pip install xyt-gps` suffira et ce bloc n'aura aucun effet.
for root in (Path.cwd(), *Path.cwd().parents):
    candidates = [root / "package-xyt-gps" / "src", root / "src"]
    for candidate in candidates:
        if (candidate / "xyt_gps").exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

import xyt_gps as xyt


def find_demo_data_dir() -> Path:
    env_path = os.environ.get("XYT_GPS_DEMO_DATA_DIR")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    for root in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([
            root / "Data" / "Output" / "anonymous-test-set-gps",
            root / "anonymous-test-set-gps",
        ])
    for candidate in candidates:
        if (candidate / "storyline.csv").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Jeu de démonstration introuvable. "
        "Définir XYT_GPS_DEMO_DATA_DIR ou demander le dossier anonymous-test-set-gps à contact@situee.ch."
    )


DATA_DIR = find_demo_data_dir()
pd.Series({"data_dir": DATA_DIR.name}, name="environnement")

## 2. Charger les tables disponibles

In [ ]:
TABLE_NAMES = [
    "storyline",
    "trips",
    "journeys",
    "user_statistics",
    "sociodemographics",
    "user_expe",
]

tables = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in TABLE_NAMES}

volume = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
            "first_columns": ", ".join(table.columns[:6]),
        }
        for name, table in tables.items()
    ]
)

display(volume)

## 3. Vérifier le contrat de colonnes

Les colonnes `required` bloquent le pipeline si elles manquent. Les colonnes `recommended` ne bloquent pas toujours, mais elles conditionnent certains enrichissements.

In [ ]:
expected_schema = xyt.expected_gps_schema()
column_report = xyt.check_raw_import_columns(
    tables["storyline"],
    tables["user_statistics"],
    trips=tables["trips"],
    journeys=tables["journeys"],
    include_recommended=True,
)

schema_summary = (
    column_report
    .groupby(["table", "status"], dropna=False)
    .size()
    .rename("columns")
    .reset_index()
)
blocking_columns = column_report.loc[column_report["status"].eq("missing_required")]

print("Colonnes attendues par table")
display(expected_schema.groupby(["table", "status"]).size().rename("columns").reset_index())
print("Contrôle des colonnes observées")
display(schema_summary)
if blocking_columns.empty:
    print("Aucune colonne obligatoire manquante.")
else:
    display(blocking_columns[["table", "expected_column", "message"]])

## 4. Vérifier les erreurs et avertissements de schéma

In [ ]:
raw = xyt.RawGpsData(
    storyline=tables["storyline"],
    trips=tables["trips"],
    journeys=tables["journeys"],
    user_statistics=tables["user_statistics"],
)
validation = xyt.validate_gps_raw(raw)

validation_summary = pd.DataFrame(
    [
        {
            "table": name,
            "ok": report.ok,
            "rows": report.n_rows,
            "columns": report.n_columns,
            "errors": sum(issue.severity == "error" for issue in report.issues),
            "warnings": sum(issue.severity == "warning" for issue in report.issues),
            "missing_required": ", ".join(report.missing_required),
            "missing_recommended": ", ".join(report.missing_recommended[:5]),
        }
        for name, report in validation.items()
    ]
)
issue_frames = [report.to_frame() for report in validation.values() if not report.to_frame().empty]
issues = pd.concat(issue_frames, ignore_index=True) if issue_frames else pd.DataFrame(
    columns=["table", "severity", "column", "message"]
)

display(validation_summary)
display(issues)

## 5. Vérifier la couverture utilisateurs

Les tables latérales peuvent avoir un périmètre plus petit que `storyline`. Ce n'est pas forcément une erreur, mais cela doit être visible avant les analyses.

In [ ]:
storyline_users = set(tables["storyline"]["user_id"].dropna().astype(str))
coverage_rows = []
for name, table in tables.items():
    if "user_id" not in table.columns:
        continue
    users = set(table["user_id"].dropna().astype(str))
    coverage_rows.append(
        {
            "table": name,
            "users": len(users),
            "missing_from_storyline": len(users - storyline_users),
            "storyline_users_missing_here": len(storyline_users - users),
        }
    )
coverage = pd.DataFrame(coverage_rows).sort_values("table")
display(coverage)

## 6. Simuler une erreur fréquente

Ici on enlève volontairement la colonne `geometry` pour voir le type de rapport attendu. Dans un vrai projet, on corrige la table ou le mapping de colonnes ; on n'ajoute pas de fallback silencieux.

In [ ]:
broken_storyline = tables["storyline"].drop(columns=["geometry"])
broken_report = xyt.check_raw_import_columns(
    broken_storyline,
    tables["user_statistics"],
    trips=tables["trips"],
    journeys=tables["journeys"],
)

display(
    broken_report.loc[broken_report["status"].eq("missing_required"), [
        "table",
        "expected_column",
        "status",
        "message",
    ]]
)

## 7. Lancer une transformation minimale

Une fois les contrôles passés, on transforme 3 utilisateurs complets. Le but est de vérifier que les dates, géométries, liens trips/journeys et mappings passent réellement.

In [ ]:
diagnostic_users = sorted(tables["storyline"]["user_id"].dropna().astype(str).unique())[:3]
sample_tables = {}
for name, table in tables.items():
    if "user_id" in table.columns:
        sample_tables[name] = table.loc[table["user_id"].astype(str).isin(diagnostic_users)].copy()
    else:
        sample_tables[name] = table.copy()

config = xyt.ProjectConfig(
    experiment_name="anonymous-test-set-gps",
    motiontag_project_name="test-project-gps",
    start_expe="2025-04-14",
    end_expe="2025-06-15",
    phases=(
        xyt.Phase("Phase1", "2025-04-14", "2025-05-04"),
        xyt.Phase("Phase2", "2025-05-05", "2025-06-01"),
        xyt.Phase("Phase3", "2025-06-02", "2025-06-15"),
    ),
    reference_year=2025,
)

raw_sample = xyt.RawGpsData(
    storyline=sample_tables["storyline"],
    trips=sample_tables["trips"],
    journeys=sample_tables["journeys"],
    user_statistics=sample_tables["user_statistics"],
)

result = xyt.run_mobility_pipeline(
    config,
    raw=raw_sample,
    sociodemo=sample_tables["sociodemographics"],
    resample_missing_days=False,
    clean_leg_geometries=False,
    add_length_outlier_flags=False,
    add_signal_quality_flags=False,
    compute_indicators=False,
)

prepared = xyt.mobility_dataset_tables(result.dataset)
prepared_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
        }
        for name, table in prepared.items()
    ]
)

display(prepared_summary)
display(result.dataset.legs[["user_id", "started_at", "finished_at", "mode_niv1", "length"]].head())

## Points à retenir

- Les erreurs de colonnes obligatoires doivent être corrigées dans la donnée source ou dans le landing.
- Une table latérale incomplète peut être acceptable si le périmètre est connu.
- Les phases doivent refléter ce qui existe réellement ; une phase future ne doit pas être compensée par un fallback implicite.
- Après les contrôles structurels, lancer une transformation minimale est le meilleur test de compatibilité.